In [1]:
import pandas as pd

opt_df = pd.read_csv("../data/opt_df_1.csv")
opt_df.head()

,campaign,cost,pCTR,click,pCTR_per_cost
0,327,0.00001,0.995833,1,99583.342057
1,530,0.00001,0.995833,1,99583.342057
2,446,0.00001,0.995833,1,99583.342057
3,624,0.00001,0.995833,1,99583.342057
4,559,0.00001,0.995833,1,99583.342057


In [2]:
SCALE = 1e5
opt_df['cost'] = opt_df['cost'] * SCALE
opt_df['pCTR_per_cost'] /= SCALE

Let's try to get K clicks as cheaply as possible

##  Minimize Cost-Per-Click (CPC) with a Click Target

Instead of “spend all budget to get max clicks”, try:

> **“Get at least K clicks, but as cheaply as possible.”**

### Math

Variables: $ x_i \ge 0 $ impressions on inventory (i).

* Expected clicks: $\sum_i p_i x_i$
* Spend: $\sum_i c_i x_i$

**Problem:**

$$
\min_x \sum_i c_i x_i
$$
subject to
$$
\sum_i p_i x_i \ge K,\quad x_i \ge 0
$$

This is a **linear program** → convex.

In [3]:
opt_df

,campaign,cost,pCTR,click,pCTR_per_cost
0,327,1.000000,0.995833,1,0.995833
1,530,1.000000,0.995833,1,0.995833
2,446,1.000000,0.995833,1,0.995833
3,624,1.000000,0.995833,1,0.995833
4,559,1.000000,0.995833,1,0.995833
...,...,...,...,...,...
49995,475,2951.588569,0.565095,1,0.000191
49996,103,1198.029395,0.203108,1,0.000170
49997,370,2819.520252,0.421777,0,0.000150
49998,619,1132.357574,0.145449,0,0.000128


In [5]:
import cvxpy as cp
import numpy as np

cost = opt_df["cost"].to_numpy()       # shape (n,)
pctr = opt_df["pCTR"].to_numpy()       # shape (n,)
n = len(opt_df)

x = cp.Variable(n, nonneg=True)

K = 500.0  # target expected clicks

objective = cp.Minimize(cost @ x)
constraints = [
    pctr @ x >= K
]

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.SCS)   # or ECOS

print("Min cost:", prob.value)
print("Expected clicks:", (pctr @ x).value)
print("Allocation (first 10):", x.value[:10])

Min cost: 502.1051356844336
Expected clicks: 500.00007053994824
Allocation (first 10): [8.96592895 8.96592895 8.96592895 8.96592895 8.96592895 8.96592895
 8.96592895 8.96592895 8.96592895 8.96592895]


In [9]:
x.value[:70]

array([8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
       8.96592895e+00, 8.96592895e+00, 8.96592895e+00, 8.96592895e+00,
      

Not doing integer programs here, we are allocating fractional impressions.
x[i] is number of impressions on inventory i.